In [0]:
# CELL 1 — Config (change nothing here)
import random, json
from datetime import datetime, timedelta
import pandas as pd

CATALOG  = "restaurant_catalog"
VOL_PATH = f"/Volumes/{CATALOG}/landing/raw_files"
random.seed(42)  # Fixed seed = reproducible data across runs
print(f"✅ Writing to: {VOL_PATH}")

In [0]:
# CELL 2 — Restaurants (5 UAE locations)
restaurants = [
    {"restaurant_id":"R001","name":"Spice Garden","city":"Dubai",     "location":"Dubai Marina","cuisine":"North Indian"},
    {"restaurant_id":"R002","name":"Biryani House","city":"Dubai",    "location":"JBR",         "cuisine":"Hyderabadi"},
    {"restaurant_id":"R003","name":"Tandoor Palace","city":"Abu Dhabi","location":"Corniche",   "cuisine":"Mughlai"},
    {"restaurant_id":"R004","name":"Curry Corner", "city":"Abu Dhabi","location":"Khalidiyah", "cuisine":"South Indian"},
    {"restaurant_id":"R005","name":"Masala Magic",  "city":"Sharjah", "location":"Al Majaz",   "cuisine":"Pan Indian"},
]
print(f"✅ Restaurants: {len(restaurants)}")

In [0]:
# CELL 3 — Menu Items (~25 per restaurant)
menu_templates = [
    ("Butter Chicken","Main Course",55.0),  ("Chicken Tikka Masala","Main Course",58.0),
    ("Dal Makhani","Main Course",42.0),     ("Palak Paneer","Main Course",45.0),
    ("Biryani Chicken","Main Course",65.0), ("Biryani Mutton","Main Course",75.0),
    ("Naan","Bread",12.0),                  ("Garlic Naan","Bread",15.0),
    ("Tandoori Roti","Bread",10.0),         ("Mango Lassi","Beverage",22.0),
    ("Masala Chai","Beverage",15.0),        ("Gulab Jamun","Dessert",25.0),
    ("Kulfi","Dessert",28.0),               ("Raita","Side",18.0),
    ("Chicken Tikka","Starter",52.0),       ("Seekh Kebab","Starter",55.0),
    ("Samosa","Starter",20.0),              ("Paneer Tikka","Starter",48.0),
    ("Prawn Masala","Main Course",80.0),    ("Malai Kofta","Main Course",48.0),
    ("Jeera Rice","Rice",30.0),             ("Kadai Chicken","Main Course",62.0),
    ("Rogan Josh","Main Course",68.0),      ("Fish Curry","Main Course",72.0),
    ("Pav Bhaji","Snack",32.0),             ("Aloo Paratha","Bread",20.0),
    ("Chole Bhature","Main Course",38.0),   ("Lemon Rice","Rice",28.0),
    ("Mango Kulfi","Dessert",30.0),         ("Jeera Aloo","Side",25.0),
]

menu_items, item_num, rest_menu = [], 1, {}
for r in restaurants:
    selected = random.sample(menu_templates, 25)
    for name, category, price in selected:
        item = {
            "item_id":       f"ITEM{item_num:04d}",
            "restaurant_id": r["restaurant_id"],
            "item_name":     name,
            "category":      category,
            "unit_price":    price,
            "is_available":  random.choice([True, True, True, False]),
        }
        menu_items.append(item)
        rest_menu.setdefault(r["restaurant_id"], []).append(item)
        item_num += 1
print(f"✅ Menu items: {len(menu_items)}")

In [0]:
# CELL 4 — Customers (500)
first = ["Arjun","Priya","Rahul","Ananya","Vikram","Deepika","Amit","Sneha",
         "Karthik","Pooja","Rajesh","Sunita","Mohammed","Fatima","Ali","Aisha",
         "Omar","Sara","Hassan","Layla","John","Emma","Michael","Olivia","James"]
last  = ["Sharma","Patel","Gupta","Singh","Kumar","Verma","Rao","Nair",
         "Khan","Ahmed","Ali","Hassan","Smith","Johnson","Williams","Brown"]
cities = ["Dubai","Abu Dhabi","Sharjah","Dubai","Dubai","Abu Dhabi"]

customers = [
    {
        "customer_id":  f"C{i:05d}",
        "name":         f"{random.choice(first)} {random.choice(last)}",
        "email":        f"customer{i}@email.com",
        "phone":        f"+971-5{random.randint(0,8)}-{random.randint(1000000,9999999)}",
        "city":         random.choice(cities),
        "loyalty_tier": random.choice(["Bronze","Silver","Gold","Platinum"]),
        "join_date":    (datetime(2023,1,1)+timedelta(days=random.randint(0,730))).strftime("%Y-%m-%d"),
    }
    for i in range(1, 501)
]
print(f"✅ Customers: {len(customers)}")

In [0]:
# CELL 5 — Historical Orders (8000, last 6 months)
order_statuses  = ["completed","completed","completed","completed","pending","cancelled"]
payment_methods = ["card","cash","wallet"]
order_types     = ["dine_in","takeaway","delivery"]
start_date      = datetime.now() - timedelta(days=180)

historical_orders = []
for oid in range(1, 8001):
    r         = random.choice(restaurants)
    c         = random.choice(customers)
    order_ts  = start_date + timedelta(
        days=random.randint(0,179), hours=random.randint(10,22), minutes=random.randint(0,59)
    )
    avail     = [m for m in rest_menu.get(r["restaurant_id"],[]) if m["is_available"]]
    chosen    = random.sample(avail, min(random.randint(1,4), len(avail))) if avail else []
    items_json = json.dumps([{
        "item_id":   it["item_id"], "item_name": it["item_name"], "category": it["category"],
        "quantity":  random.randint(1,3), "unit_price": it["unit_price"],
        "subtotal":  round(it["unit_price"] * random.randint(1,3), 2)
    } for it in chosen])
    historical_orders.append({
        "order_id":       f"ORD{oid:07d}",
        "timestamp":      order_ts.strftime("%Y-%m-%d %H:%M:%S"),
        "restaurant_id":  r["restaurant_id"],
        "customer_id":    c["customer_id"],
        "items":          items_json,
        "total_amount":   round(sum(it["unit_price"]*random.randint(1,3) for it in chosen), 2) if chosen else 0.0,
        "payment_method": random.choice(payment_methods),
        "order_type":     random.choice(order_types),
        "order_status":   random.choice(order_statuses),
    })
print(f"✅ Historical orders: {len(historical_orders)}")

In [0]:
# CELL 6 — Reviews (~40% of orders get a review)
review_texts = {
    5: ["Absolutely amazing! Best Indian restaurant in UAE.",
        "Exceptional service and authentic flavors! Will return.",
        "The biryani was outstanding. Perfect dining experience!"],
    4: ["Really good food and friendly staff. Worth visiting.",
        "Great flavors, slightly expensive but quality is there.",
        "Nice ambiance and generous portions. Would recommend."],
    3: ["Decent food but nothing extraordinary. Average experience.",
        "Food was okay, service could be better. Reasonable price.",
        "Average overall. Not bad but not special either."],
    2: ["Disappointed with the portion sizes. Prices too high.",
        "Service was slow and food was lukewarm when arrived.",
        "Not as good as expected. Probably won't return."],
    1: ["Very bad experience. Food was cold and tasteless.",
        "Terrible service. Waited 45 minutes for a simple order.",
        "Overpriced and underwhelming. Very disappointed overall."],
}

reviews, sampled = [], random.sample(historical_orders, int(len(historical_orders)*0.4))
for i, order in enumerate(sampled, 1):
    rating = random.choices([1,2,3,4,5], weights=[5,10,20,35,30])[0]
    reviews.append({
        "review_id":     f"REV{i:06d}",
        "order_id":      order["order_id"],
        "customer_id":   order["customer_id"],
        "restaurant_id": order["restaurant_id"],
        "rating":        rating,
        "review_text":   random.choice(review_texts[rating]),
        "review_date":   order["timestamp"][:10],
    })
print(f"✅ Reviews: {len(reviews)}")

In [0]:
# CELL 7 — Write All Datasets to UC Volume as CSV
datasets = {
    "restaurants":       restaurants,
    "customers":         customers,
    "menu_items":        menu_items,
    "historical_orders": historical_orders,
    "reviews":           reviews,
}

for name, data in datasets.items():
    df   = spark.createDataFrame(pd.DataFrame(data))
    path = f"{VOL_PATH}/{name}/"
    df.coalesce(1).write.mode("overwrite").option("header","true").csv(path)
    count = spark.sql(f"SELECT COUNT(*) AS cnt FROM delta.`{VOL_PATH}/{name}`").first().cnt if False else len(data)
    print(f"✅ {name:<20} {len(data):>6,} rows  →  {path}")

print("\n✅ ALL DATASETS WRITTEN. Proceed to validate.")

In [0]:
# CELL 8 — Validation (run this to confirm all files landed)
print("=== VOLUME CONTENTS ===")
print(VOL_PATH)
files = dbutils.fs.ls(VOL_PATH)
for f in files:
    print(f"  📁 {f.name:<25}  {f.size} bytes")

print(f"\n✅ {len(files)} dataset folders found in Volume")

In [0]:
# ─── CELL 8 (UPDATED) — Deep Validation ──────────────────────────────────────
#
# WHAT YOU'RE DOING:
#   dbutils.fs.ls() only shows the TOP-LEVEL folder size = always 0 bytes.
#   To see actual file sizes, you need to list INSIDE each folder.
#   This loops through each dataset folder and shows:
#   - The actual CSV part file name
#   - Its real size in KB
# ──────────────────────────────────────────────────────────────────────────────

print("=== VOLUME CONTENTS (with actual file sizes) ===\n")
folders = dbutils.fs.ls(VOL_PATH)

for folder in folders:
    print(f"📁 {folder.name}")
    inner_files = dbutils.fs.ls(folder.path)
    for f in inner_files:
        size_kb = round(f.size / 1024, 1)
        # Only show actual data files (part-*.csv), skip Spark metadata files
        if f.name.endswith(".csv"):
            print(f"   ✅ {f.name:<60}  {size_kb:>8.1f} KB  ← actual data")
        else:
            print(f"   📄 {f.name:<60}  {size_kb:>8.1f} KB  (spark metadata)")
    print()

print(f"✅ {len(folders)} dataset folders confirmed in Volume")